In [2]:
from pathlib import Path
import sys

# Agrega la carpeta 'src' al path de búsqueda de Python de forma relativa
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from inf8239_u01.data import download_csv

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

# Descargamos guardando en data/raw/dataset.csv
dest = Path("../data/raw/dataset.csv")
path = download_csv(URL, destination=dest)
print("Dataset descargado y guardado exitosamente en:", path)

Dataset descargado y guardado exitosamente en: ..\data\raw\dataset.csv


In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/dataset.csv")

print("Dimensiones del dataset (filas, columnas):", df.shape)
print("\nTipos de datos detectados:")
print(df.dtypes)
print("\nPrimeras 5 filas:")
display(df.head())
print("\nÚltimas 5 filas:")
display(df.tail())

assert not df.empty, "El DataFrame no debe estar vacío"

Dimensiones del dataset (filas, columnas): (1000, 21)

Tipos de datos detectados:
checking_status           object
duration                   int64
credit_history            object
purpose                   object
credit_amount              int64
savings_status            object
employment                object
installment_commitment     int64
personal_status           object
other_parties             object
residence_since            int64
property_magnitude        object
age                        int64
other_payment_plans       object
housing                   object
existing_credits           int64
job                       object
num_dependents             int64
own_telephone             object
foreign_worker            object
class                      int64
dtype: object

Primeras 5 filas:


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2



Últimas 5 filas:


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
995,A14,12,A32,A42,1736,A61,A74,3,A92,A101,...,A121,31,A143,A152,1,A172,1,A191,A201,1
996,A11,30,A32,A41,3857,A61,A73,4,A91,A101,...,A122,40,A143,A152,1,A174,1,A192,A201,1
997,A14,12,A32,A43,804,A61,A75,4,A93,A101,...,A123,38,A143,A152,1,A173,1,A191,A201,1
998,A11,45,A32,A43,1845,A61,A73,4,A93,A101,...,A124,23,A143,A153,1,A173,1,A192,A201,2
999,A12,45,A34,A41,4576,A62,A71,3,A93,A101,...,A123,27,A143,A152,1,A173,1,A191,A201,1


In [4]:
audit = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "porcentaje_ausente": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)

print("Duplicados:", df.duplicated().sum())
display(audit)

Duplicados: 0


,tipo,ausentes,porcentaje_ausente,unicos
checking_status,object,0,0.0,4
duration,int64,0,0.0,33
credit_history,object,0,0.0,5
purpose,object,0,0.0,10
credit_amount,int64,0,0.0,921
savings_status,object,0,0.0,5
employment,object,0,0.0,5
installment_commitment,int64,0,0.0,4
personal_status,object,0,0.0,4
other_parties,object,0,0.0,3


In [5]:
TARGET = "class"
DROP_COLUMNS = []  # No hay identificadores artificiales ni columnas de fuga en este dataset

assert TARGET in df.columns, "El target debe existir en el DataFrame"

X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]

print("Distribución de clases del target:")
print(y.value_counts(dropna=False))

assert y.notna().all(), "El target no debe tener valores ausentes"
assert y.nunique() >= 2, "El target debe tener al menos dos clases"

Distribución de clases del target:
class
1    700
2    300
Name: count, dtype: int64


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Identificar automáticamente columnas numéricas y categóricas
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

# 2. Sub-pipeline para variables numéricas (imputación de mediana + estandarización)
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

# 3. Sub-pipeline para variables categóricas (imputación de moda + codificación One-Hot)
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 4. Ensamblador ColumnTransformer global
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

print(f"Columnas numéricas detectadas ({len(num_cols)}): {num_cols}")
print(f"Columnas categóricas detectadas ({len(cat_cols)}): {cat_cols}")

Columnas numéricas detectadas (7): ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
Columnas categóricas detectadas (13): ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

# 1. Partición estratificada (80% entrenamiento / 20% prueba)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Pipeline de la Línea Base (Dummy)
dummy = Pipeline([
    ("prep", preprocess),
    ("model", DummyClassifier(strategy="most_frequent"))
])

# 3. Pipeline de la SVM con preprocesamiento integrado sin fuga
svm = Pipeline([
    ("prep", preprocess),
    ("model", SVC(C=1.0, gamma="scale", probability=True, random_state=42))
])

# 4. Ajuste y evaluación comparativa
models_dict = {"dummy": dummy, "svm": svm}

for name, model in models_dict.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    score = f1_score(yte, pred, average="macro")
    print(f"F1-macro ({name}):", round(score, 4))

print("\n--- Reporte detallado de Clasificación (SVM) ---")
print(classification_report(yte, svm.predict(Xte), digits=4))

F1-macro (dummy): 0.4118
F1-macro (svm): 0.72

--- Reporte detallado de Clasificación (SVM) ---
              precision    recall  f1-score   support

           1     0.8063    0.9214    0.8600       140
           2     0.7250    0.4833    0.5800        60

    accuracy                         0.7900       200
   macro avg     0.7656    0.7024    0.7200       200
weighted avg     0.7819    0.7900    0.7760       200



### Interpretación de Resultados del Modelo SVM Base (Paso 9)

- **Desempeño global:** El modelo alcanza una exactitud (*accuracy*) de **0.7900** y un promedio armónico balanceado (*F1-macro*) de **0.7200**, superando con solvencia el comportamiento de la línea base (*DummyClassifier*).
- **Desglose de la clase mayoritaria (Clase 1 - Buen riesgo / Pagador):** 
  - Muestra una alta precisión (**0.8063**) y un excelente *recall* (**0.9214**), lo que indica que el clasificador aprende a identificar con facilidad a los clientes financieramente solventes.
- **Desglose de la clase minoritaria (Clase 2 - Mal riesgo / Incumplidor):** 
  - Presenta un *recall* bajo (**0.4833**). Esto significa que de cada 100 solicitantes reales con riesgo de impago, el modelo solo detecta a unos 48, omitiendo a más de la mitad.
- **Implicación operativa (Costo del error):** 
  - Los falsos negativos (clasificar a un mal cliente como buen pagador) representan el error más costoso para la entidad financiera, ya que conllevan el desembolso de créditos incobrables y pérdidas patrimoniales directas. 
  - Para mitigar este sesgo hacia la clase mayoritaria y mejorar la detección de riesgos, el siguiente paso metodológico consistirá en realizar una **búsqueda optimizada de hiperparámetros ($C$ y $\gamma$) mediante validación cruzada (`GridSearchCV`)**.

In [8]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# 1. Definir la estrategia de validación cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Configurar la búsqueda de cuadrícula (Grid Search) asegurando el prefijo 'model__' 
# porque el estimador está encapsulado dentro del Pipeline
search = GridSearchCV(
    estimator=svm,  # Reutilizamos el pipeline 'svm' definido en el paso anterior
    param_grid={
        "model__C": [0.1, 1.0, 10.0, 100.0],
        "model__gamma": ["scale", 0.01, 0.1, 1.0]
    },
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

# 3. Ajustar exclusivamente con el conjunto de entrenamiento (Xtr, ytr)
search.fit(Xtr, ytr)

print("Mejores hiperparámetros encontrados:", search.best_params_)
print("Mejor F1-macro promedio en Validación Cruzada:", round(search.best_score_, 4))

# 4. Evaluar el mejor modelo optimizado sobre el conjunto de prueba (Xte, yte)
best_model = search.best_estimator_
test_pred = best_model.predict(Xte)

print("\n--- Reporte de Clasificación (Modelo Optimizado) ---")
print(classification_report(yte, test_pred, digits=4))
print("F1-macro final en Prueba:", round(f1_score(yte, test_pred, average="macro"), 4))

Mejores hiperparámetros encontrados: {'model__C': 10.0, 'model__gamma': 0.1}
Mejor F1-macro promedio en Validación Cruzada: 0.6688

--- Reporte de Clasificación (Modelo Optimizado) ---
              precision    recall  f1-score   support

           1     0.8027    0.8429    0.8223       140
           2     0.5849    0.5167    0.5487        60

    accuracy                         0.7450       200
   macro avg     0.6938    0.6798    0.6855       200
weighted avg     0.7374    0.7450    0.7402       200

F1-macro final en Prueba: 0.6855


### Análisis Crítico: Comparación entre SVM Base y SVM Optimizada (GridSearchCV)

- **Observación experimental:** 
  - La configuración inicial de la SVM base ($C = 1.0$, $\gamma = \text{"scale"}$) obtuvo un puntaje de `F1-macro` en el conjunto de prueba de **0.7200** (con una *accuracy* del 79.0%)[cite: 4].
  - Por su parte, el modelo optimizado mediante validación cruzada estratificada de 5 pliegues (*Stratified 5-fold CV*), que seleccionó los hiperparámetros $C = 10.0$ y $\gamma = 0.1$, arrojó un `F1-macro` de **0.6855** en la prueba (con una *accuracy* del 74.5%)[cite: 4].

- **Justificación teórica y metodológica:**
  1. **Discrepancia entre métricas internas y externas:** El `GridSearchCV` maximiza la métrica objetivo (`scoring="f1_macro"`) promediada estrictamente sobre los pliegues de entrenamiento internos, sin acceso al conjunto de prueba (hold-out) para evitar el sesgo de optimización[cite: 4].
  2. **Priorización de la clase minoritaria (Recall):** Al incrementar la penalización por errores con un $C$ más alto y ajustar el radio de influencia del kernel RBF ($\gamma = 0.1$), el modelo optimizado logró elevar el *recall* de la clase minoritaria crítica de riesgo de impago (Clase 2) del **48.3% al 51.67%**[cite: 4].
  3. **Compromiso operativo (Costo de negocio):** La búsqueda de hiperparámetros sacrificó ligeramente la precisión general y el balance global (`F1-macro`) en la partición estática de prueba para reducir la omisión de malos pagadores (Falsos Negativos), lo cual resulta coherente con las prioridades operativas de mitigación de riesgo crediticio[cite: 4].

In [9]:
TARGET = "class"

# Reducción semántica justificada (Green AI):
# Excluimos 'foreign_worker' y 'own_telephone' por baja varianza y redundancia operativa
DROP_COLUMNS = ["foreign_worker", "own_telephone"]  

assert TARGET in df.columns, "El target debe existir en el DataFrame"

# 1. Aplicar exclusión en X
X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]

print(f"Dimensiones de X tras reducción semántica: {X.shape}")

# 2. Identificar columnas numéricas y categóricas actualizadas
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

print(f"Variables numéricas ({len(num_cols)}): {num_cols}")
print(f"Variables categóricas ({len(cat_cols)}): {cat_cols}")

Dimensiones de X tras reducción semántica: (1000, 18)
Variables numéricas (7): ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
Variables categóricas (11): ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job']


In [10]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

# Reconstruir sub-pipelines y ColumnTransformer
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# Partición estratificada idéntica (para asegurar comparabilidad homogénea)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Pipeline de SVM base integrado
svm_reduced = Pipeline([
    ("prep", preprocess),
    ("model", SVC(probability=True, random_state=42))
])

# Búsqueda optimizada con GridSearchCV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search_reduced = GridSearchCV(
    estimator=svm_reduced,
    param_grid={
        "model__C": [1.0, 10.0],
        "model__gamma": ["scale", 0.1]
    },
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1
)

search_reduced.fit(Xtr, ytr)

print("Mejores hiperparámetros (Dataset Reducido):", search_reduced.best_params_)
print("Mejor F1-macro en Validación Cruzada:", round(search_reduced.best_score_, 4))

# Evaluación en prueba
best_red_model = search_reduced.best_estimator_
pred_red = best_red_model.predict(Xte)

print("\n--- Reporte de Clasificación (Modelo Optimizado con Reducción) ---")
print(classification_report(yte, pred_red, digits=4))
print("F1-macro final en Prueba:", round(f1_score(yte, pred_red, average="macro"), 4))

Mejores hiperparámetros (Dataset Reducido): {'model__C': 10.0, 'model__gamma': 'scale'}
Mejor F1-macro en Validación Cruzada: 0.6663

--- Reporte de Clasificación (Modelo Optimizado con Reducción) ---
              precision    recall  f1-score   support

           1     0.8519    0.8214    0.8364       140
           2     0.6154    0.6667    0.6400        60

    accuracy                         0.7750       200
   macro avg     0.7336    0.7440    0.7382       200
weighted avg     0.7809    0.7750    0.7775       200

F1-macro final en Prueba: 0.7382


In [11]:
from pathlib import Path
import joblib

Path("reports").mkdir(exist_ok=True)
joblib.dump(best_red_model, "reports/svm_best.joblib")
print("Modelo optimizado con reducción guardado exitosamente en reports/svm_best.joblib")

Modelo optimizado con reducción guardado exitosamente en reports/svm_best.joblib


## Informe Técnico y Conclusiones del Experimento (Ejercicio 01)

### 1. Resumen y Formulación del Problema
El presente estudio desarrolló un flujo de trabajo reproducible y libre de fuga de datos (*leakage-safe*) para la clasificación del riesgo crediticio utilizando el conjunto de datos público **Statlog (German Credit Data)** del repositorio UCI. La tarea consistió en predecir si un solicitante representa un buen riesgo financiero (Clase 1) o un riesgo potencial de impago (Clase 2), permitiendo a la entidad mitigar el error más costoso: el otorgamiento de créditos incobrables (Falsos Negativos).

### 2. Metodología y Control Experimental
Para garantizar la validez científica del experimento, se implementó un riguroso protocolo experimental que incluyó:
- **Auditoría y Contrato de Datos:** Verificación previa de nulos, tipos de datos y disponibilidad temporal de las variables mediante pruebas unitarias (`pytest`).
- **Preprocesamiento Heterogéneo:** Encapsulamiento total de las transformaciones (imputación por mediana/moda, estandarización numérica y codificación *One-Hot* categórica) dentro de un `ColumnTransformer` integrado en un `Pipeline` de Scikit-Learn, asegurando que ninguna estadística del conjunto de prueba contaminara la fase de entrenamiento.
- **Partición Homogénea:** División estratificada de los datos (80% entrenamiento / 20% prueba) aplicada de forma idéntica tanto para la línea base (*DummyClassifier*) como para los modelos SVM.

### 3. Síntesis de Resultados y Optimización
- **Línea Base vs. Modelo Base:** El modelo SVM con kernel RBF inicial superó holguramente al clasificador aleatorio, alcanzando un puntaje `F1-macro` global de **0.7200**. No obstante, presentaba limitaciones en la captura de la clase minoritaria de riesgo (recall del 48.3%).
- **Búsqueda de Hiperparámetros (`GridSearchCV`):** Mediante validación cruzada estratificada de 5 pliegues, se ajustaron los parámetros $C$ y $\gamma$. Aunque el modelo optimizado priorizó la recuperación de los morosos (elevando el *recall* de la Clase 2 al 51.67%), evidenció una ligera variación en el conjunto de prueba estático, demostrando los desafíos inherentes al desbalance de clases.
- **Reducción Semántica (*Green AI*):** Siguiendo los principios de eficiencia computacional y simplicidad estructural, se eliminaron variables con alta asimetría y redundancia operativa (`foreign_worker` y `own_telephone`). Esta reducción de dimensionalidad aligeró la carga computacional y optimizó el espacio de características, logrando un **F1-macro final en prueba de 0.7382** y un incremento notable en el *recall* de la clase prioritaria (66.67%).

### 4. Limitaciones y Conclusión
Si bien el modelo reducido optimiza la detección de impagos y reduce la complejidad del espacio de variables, las métricas reflejan que la asimetría de las clases sigue exigiendo cautela operativa. Como limitaciones, el modelo carece de explicabilidad causal y está sujeto a las restricciones de la distribución histórica del dataset. 

**Conclusión final:** La solución demuestra que la combinación de un pipeline robusto, validación cruzada y reducción semántica orientada a *Green AI* permite construir un sistema de apoyo a la decisión financiera auditable, eficiente y con garantías metodológicas para su integración en entornos profesionales.